# Ejercicios de XML con ElementTree

**Acceso a Datos · Tema 1: Manejo de ficheros** · *Versión sin soluciones*

> ⚠️ **Contenido generado con ayuda de IA.** Los enunciados y las soluciones se han generado con ayuda de una
> herramienta de inteligencia artificial y se han ejecutado antes de publicarlos, pero pueden contener errores.
> Contrasta lo que leas con la [documentación oficial de ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html).

Diez ejercicios en orden creciente de dificultad. Solo se usa el módulo `xml.etree.ElementTree` y otros módulos de la
biblioteca estándar (`pathlib`, `csv`, `json`, `zipfile`, `dataclasses`, `urllib`…). No hace falta instalar nada.

| Nº | Ejercicio | Dificultad | Lo que se practica |
|---|---|:---:|---|
| 1 | Inventario de un taller de bicicletas | ★ | `parse`, `getroot`, `find`, `findtext`, atributos |
| 2 | Consultas a un horario de trenes | ★ | XPath de ElementTree y sus límites |
| 3 | Clasificación de una liga | ★ | Crear documentos: `Element`, `SubElement`, `indent`, `write` |
| 4 | Limpieza de una lista de reproducción | ★★ | Modificar el árbol: `remove`, `set`, `.tag`, ordenar |
| 5 | Lecturas de sensores con datos sucios | ★★ | Conversión robusta y `ParseError` |
| 6 | Ruta de senderismo en GPX | ★★ | Espacios de nombres |
| 7 | Fusión de pedidos de dos sucursales | ★★ | Transformar una jerarquía en otra |
| 8 | Mirar dentro de un `.docx` | ★★★ | XML real dentro de un ZIP, varios espacios de nombres |
| 9 | Un fichero XML enorme | ★★★ | `iterparse` frente a `parse`: memoria y tiempo |
| 10 | Proyecto: el tiempo en Castilla y León | ★★★ | Datos reales de AEMET: descarga, caché, codificación, modelo |

**Estructura de carpetas.** El cuaderno espera esta organización (los ficheros de `salida/` los crean los ejercicios):

```
ejercicios_xml/
├── Ejercicios_XML_ElementTree.ipynb
├── datos/
│   ├── taller.xml, horario.xml, playlist.xml
│   ├── estacion.xml, estacion_rota.xml
│   ├── ruta.gpx, pedidos_norte.xml, pedidos_sur.xml, memoria.docx
│   └── aemet/localidad_47186.xml      ← copia de la predicción real de Valladolid
└── salida/
```

Ejecuta primero la celda siguiente: importa ElementTree y define las rutas que usan todos los ejercicios.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

DATOS = Path("datos")
SALIDA = Path("salida")
SALIDA.mkdir(exist_ok=True)

---
## Ejercicio 1. Inventario de un taller de bicicletas ★

El fichero `datos/taller.xml` guarda el inventario de un taller. Cada bicicleta es un elemento `<bici>` con los
atributos `id` y `tipo`, y los elementos `marca`, `modelo`, `precio` y `stock`:

```xml
<bici id="B001" tipo="montaña">
  <marca>Orbea</marca>
  <modelo>Alma M30</modelo>
  <precio moneda="EUR">1899.00</precio>
  <stock>3</stock>
</bici>
```

Escribe un programa que:

1. Muestre el nombre y la ciudad del taller, que son atributos del elemento raíz.
2. Muestre una tabla alineada con el id, tipo, marca, modelo, precio (con dos decimales, alineado a la derecha) y stock
   de cada bici.
3. Si una bici no tiene atributo `tipo`, muestre «sin clasificar».
4. Si una bici no tiene elemento `stock`, lo cuente como 0 **y avise al final** de qué bicis no tienen el dato.
5. Calcule el valor total del inventario (precio × stock) y diga cuál es la bici más cara.

**Pistas.** `elemento.get("atributo", valor_por_defecto)` y `elemento.findtext("hijo", valor_por_defecto)`.
Piensa si te conviene usar el valor por defecto de `findtext` en el apartado 4.

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 2. Consultas a un horario de trenes ★

El fichero `datos/horario.xml` contiene los trenes que salen de Valladolid un día:

```xml
<tren numero="18021" tipo="MD">
  <destino>Salamanca</destino>
  <paradas>
    <parada><estacion>Valladolid-Campo Grande</estacion><hora>07:40</hora></parada>
    <parada><estacion>Medina del Campo</estacion><hora>08:12</hora></parada>
    ...
  </paradas>
  <servicios/>            <!-- puede tener <cafeteria/> y <wifi/>, o no existir -->
</tren>
```

Resuelve cada consulta **con una sola llamada a `findall` o `find`** siempre que se pueda, y con Python cuando
ElementTree no lo permita:

1. Número y hora de salida de los trenes de tipo AVE.
2. Trenes que paran en Medina del Campo.
3. Última parada de cada tren y hora de llegada.
4. Trenes con cafetería.
5. Trenes **sin** cafetería.
6. El tercer tren del horario.
7. Lista de todas las estaciones distintas, ordenada alfabéticamente.

Prueba también `horario.findall(".//tren[servicios/cafeteria]")` y explica qué ocurre.

**Resumen del XPath que admite ElementTree**

| Expresión | Selecciona |
|---|---|
| `tag`, `*`, `.`, `..`, `//` | hijos con esa etiqueta, todos los hijos, el propio nodo, el padre, cualquier descendiente |
| `[@atr]`, `[@atr='v']`, `[@atr!='v']` | elementos con ese atributo, con ese valor o con otro valor |
| `[tag]` | elementos que tienen un **hijo directo** `tag` |
| `[tag='texto']`, `[.='texto']` | hijo (o el propio elemento) cuyo texto completo es `texto` |
| `[1]`, `[last()]`, `[last()-1]` | posición entre los hermanos con la misma etiqueta (empieza en 1) |

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 3. Clasificación de una liga ★

Tienes los resultados de las tres primeras jornadas de una liga provincial en una lista de Python (ejecuta la
celda siguiente). Genera el fichero `salida/clasificacion.xml` con la clasificación.

1. Calcula para cada equipo: partidos jugados, ganados, empatados y perdidos, goles a favor y en contra, diferencia de
   goles y puntos (3 por victoria, 1 por empate).
2. Ordena por puntos, después por diferencia de goles, después por goles a favor y, en último caso, por nombre.
3. Diseña la estructura del XML: decide qué datos van como **atributo** y cuáles como **elemento**, y justifícalo en
   una celda de texto.
4. El fichero debe llevar declaración XML, estar en UTF-8 y estar sangrado.
5. Vuelve a leer el fichero generado y muestra los tres primeros.

**Pistas.** `ET.Element`, `ET.SubElement`, `ET.indent(arbol)` y
`arbol.write(ruta, encoding="utf-8", xml_declaration=True)`. Los valores de atributos y textos deben ser cadenas.

In [ ]:
RESULTADOS = [
    {"jornada": 1, "local": "CD Pinares",    "visitante": "Atlético Duero", "gl": 2, "gv": 1},
    {"jornada": 1, "local": "UD Meseta",     "visitante": "CF Castillo",    "gl": 0, "gv": 0},
    {"jornada": 1, "local": "Racing Páramo", "visitante": "SD Ribera",      "gl": 3, "gv": 2},
    {"jornada": 2, "local": "Atlético Duero","visitante": "UD Meseta",      "gl": 1, "gv": 1},
    {"jornada": 2, "local": "CF Castillo",   "visitante": "Racing Páramo",  "gl": 2, "gv": 0},
    {"jornada": 2, "local": "SD Ribera",     "visitante": "CD Pinares",     "gl": 0, "gv": 1},
    {"jornada": 3, "local": "CD Pinares",    "visitante": "CF Castillo",    "gl": 1, "gv": 3},
    {"jornada": 3, "local": "UD Meseta",     "visitante": "Racing Páramo",  "gl": 2, "gv": 2},
    {"jornada": 3, "local": "SD Ribera",     "visitante": "Atlético Duero", "gl": 1, "gv": 0},
]

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 4. Limpieza de una lista de reproducción ★★

El fichero `datos/playlist.xml` tiene una lista de canciones con título, artista, duración (`m:ss`) y valoración
de 1 a 5. Escribe un programa que haga estos cambios **en este orden** y guarde el resultado en
`salida/playlist_limpia.xml` (el original no se modifica):

1. Elimina las canciones de menos de 60 segundos.
2. Añade el atributo `favorita="si"` a las canciones con valoración 4 o 5.
3. Cambia la etiqueta `<cancion>` por `<pista>`.
4. Ordena las pistas por duración, de menor a mayor.
5. Añade a la raíz los atributos `total_pistas` y `duracion_total` (en formato `m:ss`).

Antes de resolverlo, ejecuta la celda de «versión con error» y explica por qué no borra todas las canciones cortas.

In [ ]:
# Versión con error: borrar mientras se recorre la lista de hijos
def segundos(texto):
    minutos, segs = texto.split(":")
    return int(minutos) * 60 + int(segs)


raiz = ET.parse(DATOS / "playlist.xml").getroot()
for cancion in raiz:
    if segundos(cancion.findtext("duracion")) < 60:
        raiz.remove(cancion)

for cancion in raiz:
    print(cancion.findtext("duracion"), cancion.findtext("titulo"))

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 5. Lecturas de sensores con datos sucios ★★

Una estación meteorológica casera guarda sus lecturas en `datos/estacion.xml`. El fichero tiene **errores a
propósito**: decimales con coma, valores vacíos o ausentes, el texto `N/D`, espacios sobrantes, fechas en dos
formatos (`2026-10-12T08:00:00` y `12/10/2026 10:00`), una lectura sin fecha y el valor `-999`, que el sensor usa
cuando falla.

1. Escribe una función `leer_lecturas(ruta)` que devuelva una lista de lecturas válidas (diccionarios con `fecha`,
   `temperatura`, `humedad` y `presion`) y una lista de incidencias con texto legible (por ejemplo,
   `"Lectura 4: falta la temperatura"`).
2. Ningún dato erróneo debe hacer que el programa falle. Un valor no válido se guarda como `None` y genera una
   incidencia; el resto de valores de esa lectura se aprovechan.
3. Considera no válidas las temperaturas fuera de [-50, 60] y las humedades fuera de [0, 100].
4. Muestra las incidencias y la media de cada magnitud indicando cuántos valores se han usado.
5. Llama a la función con `datos/estacion_rota.xml`, que está mal formado, y con un fichero que no existe. El
   programa debe mostrar un mensaje claro: en el primer caso, la línea y la columna del error y el contenido de esa línea.

**Pista.** `ET.ParseError` tiene el atributo `position`, una tupla `(línea, columna)`.

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 6. Ruta de senderismo en GPX ★★

GPX es el formato XML estándar para rutas GPS: lo exportan Wikiloc, Strava, Komoot, los relojes deportivos y casi
cualquier app de senderismo. En `datos/ruta.gpx` tienes una ruta de ejemplo (sintética) de subida a Peñalara. Si
tienes alguna ruta tuya, puedes usarla en su lugar.

```xml
<gpx version="1.1" xmlns="http://www.topografix.com/GPX/1/1" ...>
  <trk>
    <name>Peñalara desde Cotos</name>
    <trkseg>
      <trkpt lat="40.822600" lon="-3.958500">
        <ele>1830.0</ele>
        <time>2026-10-11T08:15:00Z</time>
      </trkpt>
      ...
```

1. Ejecuta `raiz.findall(".//trkpt")` y explica por qué devuelve una lista vacía. Muestra cómo se llama realmente la
   etiqueta (`raiz.tag`).
2. Lee todos los puntos usando un diccionario de espacios de nombres.
3. Calcula y muestra: nombre de la ruta, número de puntos, distancia total en km (fórmula del *haversine*), desnivel
   positivo y negativo, altitud máxima y mínima, duración y velocidad media.
4. Exporta los puntos a `salida/ruta.csv` con las columnas `lat`, `lon`, `ele`, `hora` y `km` (distancia acumulada).
5. *(Ampliación)* Obtén el espacio de nombres a partir de `raiz.tag` en lugar de escribirlo a mano, para que el
   programa funcione también con ficheros GPX 1.0.

**Fórmula del haversine** (distancia entre dos puntos sobre una esfera de radio R = 6371 km, con latitudes φ y
longitudes λ en radianes):

$$a = \sin^2\left(\frac{\Delta\varphi}{2}\right) + \cos\varphi_1 \cos\varphi_2 \sin^2\left(\frac{\Delta\lambda}{2}\right)
\qquad d = 2R \cdot \operatorname{atan2}\left(\sqrt{a}, \sqrt{1-a}\right)$$

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 7. Fusión de pedidos de dos sucursales ★★

Una empresa de suministros tiene dos sucursales y cada una exporta sus pedidos en un XML con estructura **plana**
(`datos/pedidos_norte.xml` y `datos/pedidos_sur.xml`):

```xml
<pedidos sucursal="Norte">
  <pedido id="1001" fecha="2026-10-01" cliente_id="C03" cliente="Ferretería Pisuerga">
    <producto>Taladro percutor</producto>
    <cantidad>2</cantidad>
    <precio_unitario>89.90</precio_unitario>
  </pedido>
```

Genera `salida/pedidos_por_cliente.xml`, con los pedidos **agrupados por cliente**:

```xml
<clientes generado="...">
  <cliente id="C01" nombre="Bricolaje Duero" num_pedidos="3" total="...">
    <pedido id="1002" sucursal="Norte" fecha="2026-10-02">
      <producto>...</producto><cantidad>...</cantidad><precio_unitario>...</precio_unitario><importe>...</importe>
    </pedido>
    ...
```

1. Los clientes se ordenan por `id` y sus pedidos, por fecha.
2. Cada pedido indica de qué sucursal viene y su importe (cantidad × precio unitario).
3. Detecta los pedidos con el mismo `id` en las dos sucursales. Si son idénticos, se incluyen una sola vez y se avisa.
   Si tienen datos distintos, se informa del conflicto y no se incluye ninguno.
4. Usa `decimal.Decimal` para el dinero.
5. Muestra un resumen por cliente.

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 8. Mirar dentro de un `.docx` ★★★

Un documento de Word (`.docx`), igual que un `.xlsx` o un `.odt`, es un **fichero ZIP** que contiene varios XML.
El texto está en `word/document.xml`, los estilos en `word/styles.xml` y los metadatos en `docProps/core.xml`.

Trabaja con `datos/memoria.docx` usando solo `zipfile` y ElementTree:

1. Lista los ficheros que contiene el ZIP.
2. Muestra el título y el autor del documento y su fecha de creación, que están en `docProps/core.xml`.
3. Cuenta los párrafos con texto y las palabras.
4. Muestra el índice del documento: los títulos con sangría según su nivel.
5. Guarda el texto plano en `salida/memoria.txt`, con una línea en blanco entre párrafos.

**Lo que necesitas saber del formato.** Cada párrafo es un `<w:p>`. Dentro, el texto está repartido en uno o varios
fragmentos `<w:r>` (*runs*), cada uno con su `<w:t>`: una palabra en negrita en mitad de una frase parte el texto
en tres *runs*. El estilo del párrafo está en `<w:pPr><w:pStyle w:val="Heading1"/></w:pPr>`.

**Cuidado con los estilos.** El identificador del estilo depende del idioma de Word: en un Word en español el
título 1 se llama `Ttulo1`. Para que el programa sirva con cualquier documento, busca en `word/styles.xml` el
nombre interno del estilo (`<w:name w:val="heading 1"/>`), que siempre está en inglés.

```python
NS = {
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "cp": "http://schemas.openxmlformats.org/package/2006/metadata/core-properties",
}
```

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 9. Un fichero XML enorme ★★★

`ET.parse` construye **todo el árbol en memoria** antes de devolverlo. Con ficheros de cientos de MB o varios GB,
esto puede agotar la memoria. `ET.iterparse` lee el fichero poco a poco y avisa cada vez que se abre o se cierra un
elemento, de modo que se puede procesar cada uno y liberarlo enseguida, como se hace con SAX.

1. Ejecuta la celda que genera `salida/accesos.xml`, un registro de peticiones a un servidor web:
   ```xml
   <peticion ip="10.0.3.17" fecha="2026-10-12T14:03:22" metodo="GET" codigo="200" bytes="5321">
     <ruta>/productos/23</ruta>
     <agente>Mozilla/5.0 ...</agente>
   </peticion>
   ```
   Por defecto tiene 200 000 peticiones (unos 40 MB). Si tu ordenador lo aguanta, prueba con 2 000 000 (unos 400 MB).
2. Escribe `estadisticas_parse(ruta)`, que calcule con `ET.parse`: número de peticiones por código de estado, la IP con
   más peticiones, la hora del día con más tráfico y el total de bytes servidos.
3. Escribe `estadisticas_iterparse(ruta)`, que calcule lo mismo con `ET.iterparse` y libere cada `<peticion>` en
   cuanto la haya procesado.
4. Mide el pico de memoria (`tracemalloc`) y el tiempo (`time.perf_counter`) de cada versión, comprueba que dan el
   mismo resultado y muestra una tabla comparativa.

**Pista.** `elem.clear()` vacía el elemento, pero el elemento vacío sigue colgando de la raíz. Para liberar de
verdad la memoria hay que quitarlo también de su padre, o vaciar la raíz cada cierto tiempo. Para tener una referencia
a la raíz, pide también los eventos `"start"`.

In [ ]:
import random
from datetime import datetime, timedelta


def generar_registro(ruta, n=200_000, semilla=1):
    """Escribe el XML línea a línea, sin construir el árbol: así no ocupa memoria."""
    rnd = random.Random(semilla)
    ips = [f"10.0.{rnd.randint(0, 9)}.{rnd.randint(1, 254)}" for _ in range(400)]
    rutas = ["/", "/productos", "/carrito", "/login", "/api/stock"] + [f"/productos/{i}" for i in range(1, 60)]
    agentes = ["Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Mozilla/5.0 (X11; Linux x86_64)",
               "Mozilla/5.0 (iPhone; CPU iPhone OS 18_0 like Mac OS X)", "curl/8.9.1"]
    codigos = ["200"] * 85 + ["304"] * 6 + ["404"] * 6 + ["500"] * 2 + ["403"]
    pesos_hora = [1, 1, 1, 1, 1, 2, 3, 5, 8, 9, 10, 11, 12, 10, 9, 9, 10, 12, 14, 13, 11, 8, 5, 3]
    inicio = datetime(2026, 10, 12)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n<registro servidor="web01">\n')
        for _ in range(n):
            hora = rnd.choices(range(24), weights=pesos_hora)[0]
            fecha = inicio + timedelta(hours=hora, seconds=rnd.randrange(3600))
            f.write(f'  <peticion ip="{rnd.choice(ips)}" fecha="{fecha:%Y-%m-%dT%H:%M:%S}" '
                    f'metodo="{rnd.choice(["GET"] * 9 + ["POST"])}" codigo="{rnd.choice(codigos)}" '
                    f'bytes="{rnd.randint(200, 90_000)}">\n'
                    f'    <ruta>{rnd.choice(rutas)}</ruta>\n'
                    f'    <agente>{rnd.choice(agentes)}</agente>\n'
                    f'  </peticion>\n')
        f.write("</registro>\n")


ACCESOS = SALIDA / "accesos.xml"
generar_registro(ACCESOS)
print(f"{ACCESOS}: {ACCESOS.stat().st_size / 1_000_000:.1f} MB")

In [ ]:
# Escribe aquí tu solución


---
## Ejercicio 10. Proyecto: el tiempo en Castilla y León con AEMET ★★★

AEMET publica en su web la predicción de los próximos siete días para cada municipio en XML. No hace falta
registrarse ni pedir una clave:

```
https://www.aemet.es/xml/municipios/localidad_<código>.xml
```

El código es el del INE, con cinco cifras. Es el número que aparece al final de la ficha del municipio en aemet.es:
`.../prediccion/municipios/valladolid-id47186` → `47186`.

> **Uso responsable.** Guarda las descargas en una caché, no hagas peticiones en bucle y deja una pausa entre
> municipios. El aviso legal de AEMET exige citarla como fuente de los datos.

En `datos/aemet/localidad_47186.xml` tienes una copia de la predicción real de Valladolid del 19/10/2026 para poder
trabajar sin conexión. Ábrela y fíjate en estos detalles antes de empezar:

- La codificación es **ISO-8859-15**, no UTF-8.
- Hay muchos elementos vacíos, como `<prob_precipitacion periodo="00-06"/>` o `<racha_max/>`.
- Los dos primeros días tienen datos por periodos de 6 horas (`00-06`, `06-12`…), los dos siguientes por periodos
  de 12 horas y los tres últimos no tienen atributo `periodo`.
- En el primer día, los periodos que ya han pasado están vacíos, incluido el del día completo (`00-24`).
- Algunos códigos de estado del cielo llevan una `n` de «noche», como `81n`.

### Apartados

1. **Descarga con caché.** Escribe `obtener_xml(codigo)`, que devuelva los **bytes** del XML. Si el fichero de caché
   tiene menos de una hora, lo usa; si no, lo descarga y lo guarda. Si la descarga falla y hay una copia antigua, la
   usa avisando.
2. **Codificación.** Comprueba qué pasa si decodificas los bytes como UTF-8 y explica por qué conviene pasar los
   bytes directamente a `ET.fromstring`.
3. **Modelo.** Convierte el XML en objetos: una *dataclass* `Prediccion` (código, nombre, provincia, fecha de
   elaboración y lista de días) y otra `DiaPrediccion` (fecha, máxima, mínima, probabilidad de precipitación del
   día, descripción y código del cielo, racha máxima). Los datos ausentes se guardan como `None`. Decide y justifica
   cómo obtienes el dato «del día completo» cuando `00-24` está vacío.
4. **Varias capitales.** Procesa las nueve capitales de provincia de Castilla y León y muestra una tabla con la
   predicción para **mañana** (el día siguiente a la fecha de elaboración), la capital más cálida y la más lluviosa.
5. **Resumen propio.** Genera `salida/resumen_cyl.xml` con una estructura diseñada por ti y expórtalo también a
   `salida/resumen_cyl.json` y `salida/resumen_cyl.csv`.
6. *(Ampliación)* **Histórico.** Cada vez que se ejecute, añade la predicción a `salida/historico.xml` (sin duplicar
   una misma elaboración). Escribe una función que muestre todas las predicciones guardadas para una fecha y
   municipio, para ver cómo cambia la predicción según se acerca el día.
7. *(Ampliación avanzada)* **Avisos.** Con una clave gratuita de AEMET OpenData, descarga los avisos en formato CAP de
   Castilla y León, descomprímelos con `tarfile` y lista los avisos activos.

#### 1. Descarga con caché

In [ ]:
# Escribe aquí tu solución


#### 2. Codificación

In [ ]:
# Escribe aquí tu solución


#### 3. Modelo

In [ ]:
# Escribe aquí tu solución


#### 4. Las nueve capitales

In [ ]:
# Escribe aquí tu solución


#### 5. Resumen en XML, JSON y CSV

In [ ]:
# Escribe aquí tu solución


#### 6. Ampliación: histórico

In [ ]:
# Escribe aquí tu solución


#### 7. Ampliación avanzada: avisos CAP

In [ ]:
# Escribe aquí tu solución


---
### Fuentes

- [Documentación de `xml.etree.ElementTree`](https://docs.python.org/3/library/xml.etree.elementtree.html)
- [Predicción por municipios de AEMET en XML](https://www.aemet.es/xml/municipios/localidad_47186.xml) y
  [AEMET OpenData](https://opendata.aemet.es/). Datos meteorológicos: © AEMET.
- [Formato GPX 1.1](https://www.topografix.com/GPX/1/1/)
- [Especificación CAP 1.2 (OASIS)](https://docs.oasis-open.org/emergency/cap/v1.2/CAP-v1.2.html)